# Advanced Python Dictionary Problems — With Complete Solutions

This notebook develops advanced, practical uses of Python dictionaries from the following core operations:

- `len(mapping)`
- direct lookup with `mapping[key]`
- safe lookup with `mapping.get(key, default)`
- membership tests with `in` and `not in`
- deletion with `del`, `pop`, and `popitem`
- conditional insertion and grouping with `setdefault`
- clearing with `clear`
- insertion-order behavior

The exercises emphasize **correctness, readability, explicit edge-case handling, testability, and time-complexity awareness**. Every problem includes a complete solution and executable checks.


## How to Use This Notebook

For each problem:

1. Read the specification and examples.
2. Write your own implementation before opening or running the solution cell.
3. Compare behavior on edge cases, not only the happy path.
4. Run the assertions. A silent cell means the checks passed.
5. Review the complexity and best-practice notes.

All solutions use only the Python standard library.


In [1]:
from __future__ import annotations

from collections.abc import Callable, Hashable, Iterable, Mapping, MutableMapping, Sequence
from copy import deepcopy
from dataclasses import dataclass
from math import isclose
from typing import Any, TypeVar

K = TypeVar("K", bound=Hashable)
V = TypeVar("V")
T = TypeVar("T")

MISSING = object()  # Unique sentinel: different from every valid stored value.


## Best-Practice Checklist

- Use `d[key]` when a missing key is a programming error and should fail loudly.
- Use `key in d` when presence itself matters.
- Use `d.get(key, default)` when a real default is meaningful.
- Use a unique sentinel when `None` may be a legitimate value.
- Use `pop(key, default)` for safe remove-and-return behavior.
- Use `setdefault` for concise grouping, but do not hide expensive default construction inside it.
- Avoid mutating a dictionary while iterating over its live view.
- Remember that `dict.fromkeys(keys, mutable_object)` shares the same object across every key.
- Treat `popitem()` as LIFO in modern Python; it is not an LRU-cache primitive.
- Keep transformation functions pure when possible, and test edge cases explicitly.


## Quick Reference: Core Idioms


In [2]:
settings = {"theme": "dark", "timeout": None}

# 1) Direct access: appropriate when the key must exist.
assert settings["theme"] == "dark"

# 2) get(): appropriate when a default is meaningful.
assert settings.get("language", "en") == "en"

# 3) Membership distinguishes missing from present-with-None.
assert "timeout" in settings
assert settings["timeout"] is None

# 4) Safe removal.
removed = settings.pop("language", MISSING)
assert removed is MISSING

# 5) Insert a default only if absent.
roles_by_team: dict[str, list[str]] = {}
roles_by_team.setdefault("data", []).append("analyst")
assert roles_by_team == {"data": ["analyst"]}

# 6) LIFO removal of the most recently inserted pair.
stack_like = {"first": 1, "second": 2, "third": 3}
assert stack_like.popitem() == ("third", 3)

# 7) clear() mutates the original object, preserving aliases.
original = {"a": 1}
alias = original
original.clear()
assert alias == {}


---
## Problem 1 — Distinguish a Missing Key from a Stored `None`

**Difficulty:** Intermediate  
**Core operations:** `get`, membership, sentinel object

Implement `lookup_state(mapping, key)` so it returns one of:

- `("missing", None)` when the key is absent,
- `("present-none", None)` when the key exists and stores `None`,
- `("present-value", value)` for every other stored value.

### Why this matters

`mapping.get(key)` alone cannot distinguish an absent key from a key whose value is actually `None`.

### Target complexity

Average-case **O(1)** time and **O(1)** extra space.


In [3]:
def lookup_state(mapping: Mapping[K, V], key: K) -> tuple[str, V | None]:
    """Classify a dictionary lookup without confusing missing with None."""
    value = mapping.get(key, MISSING)
    if value is MISSING:
        return ("missing", None)
    if value is None:
        return ("present-none", None)
    return ("present-value", value)


sample = {"ready": True, "payload": None, "attempts": 0}
assert lookup_state(sample, "unknown") == ("missing", None)
assert lookup_state(sample, "payload") == ("present-none", None)
assert lookup_state(sample, "ready") == ("present-value", True)
assert lookup_state(sample, "attempts") == ("present-value", 0)


**Best-practice note:** prefer a module-level sentinel such as `MISSING = object()` over a string like `"missing"`, because a user could legitimately store that string as a value.


---
## Problem 2 — Configurable Histogram Builder

**Difficulty:** Intermediate  
**Core operations:** `get`, membership, `len`

Implement `histogram(items, *, key=None, ignore=None)`.

Requirements:

- Count transformed items in a dictionary.
- `key` is an optional transformation function.
- `ignore` is an optional set of transformed values to skip.
- Preserve the insertion order of first appearance.
- Work with any hashable transformed value.

Example:

```python
histogram("Banana!", key=str.lower, ignore={"!"})
# {'b': 1, 'a': 3, 'n': 2}
```


In [4]:
def histogram(
    items: Iterable[T],
    *,
    key: Callable[[T], K] | None = None,
    ignore: set[K] | None = None,
) -> dict[K, int]:
    """Count transformed items while preserving first-seen key order."""
    transform: Callable[[T], K]
    if key is None:
        transform = lambda item: item  # type: ignore[return-value]
    else:
        transform = key

    ignored = ignore if ignore is not None else set()
    counts: dict[K, int] = {}

    for item in items:
        normalized = transform(item)
        if normalized in ignored:
            continue
        counts[normalized] = counts.get(normalized, 0) + 1

    return counts


assert histogram("Banana!", key=str.lower, ignore={"!"}) == {
    "b": 1,
    "a": 3,
    "n": 2,
}
assert histogram([1, 2, 1, 3, 2, 1]) == {1: 3, 2: 2, 3: 1}
assert histogram([], key=str) == {}


**Complexity:** O(n) average time and O(k) space, where `k` is the number of distinct retained keys.


---
## Problem 3 — Nested Grouping with `setdefault`

**Difficulty:** Intermediate  
**Core operations:** `setdefault`, membership, nested dictionaries

Given employee records, build this structure:

```python
{
    department: {
        role: [employee_name, ...]
    }
}
```

Requirements:

- Keep employee names in input order.
- Reject records missing `department`, `role`, or `name` with a clear `KeyError`.
- Do not pre-create departments or roles that never occur.


In [5]:
def group_employees(
    records: Iterable[Mapping[str, str]],
) -> dict[str, dict[str, list[str]]]:
    """Group names by department and role, preserving input order."""
    grouped: dict[str, dict[str, list[str]]] = {}

    for record in records:
        # Direct indexing is intentional: malformed records should fail loudly.
        department = record["department"]
        role = record["role"]
        name = record["name"]

        roles = grouped.setdefault(department, {})
        roles.setdefault(role, []).append(name)

    return grouped


employees = [
    {"name": "Ava", "department": "Engineering", "role": "Backend"},
    {"name": "Noah", "department": "Engineering", "role": "Frontend"},
    {"name": "Mia", "department": "Engineering", "role": "Backend"},
    {"name": "Leo", "department": "Finance", "role": "Analyst"},
]

expected = {
    "Engineering": {
        "Backend": ["Ava", "Mia"],
        "Frontend": ["Noah"],
    },
    "Finance": {"Analyst": ["Leo"]},
}
assert group_employees(employees) == expected


### Alternative without `setdefault`

The explicit version is longer but sometimes easier to debug:

```python
if department not in grouped:
    grouped[department] = {}
if role not in grouped[department]:
    grouped[department][role] = []
grouped[department][role].append(name)
```

Choose the version your team finds clearest.


---
## Problem 4 — Build an Inverted Index with Positions

**Difficulty:** Advanced  
**Core operations:** nested `setdefault`, `get`, membership

Build an inverted index from a sequence of documents.

Return:

```python
{
    word: {
        document_id: [position_0, position_1, ...]
    }
}
```

Rules:

- Tokenize by whitespace.
- Normalize words with `casefold()`.
- Strip the punctuation characters `.,!?;:` from both ends.
- Ignore empty tokens.
- Positions are zero-based token positions within each document before empty-token removal.


In [6]:
def build_inverted_index(
    documents: Mapping[str, str],
) -> dict[str, dict[str, list[int]]]:
    """Map normalized words to document IDs and token positions."""
    index: dict[str, dict[str, list[int]]] = {}

    for document_id, text in documents.items():
        for position, raw_token in enumerate(text.split()):
            word = raw_token.strip(".,!?;:").casefold()
            if not word:
                continue
            postings = index.setdefault(word, {})
            postings.setdefault(document_id, []).append(position)

    return index


documents = {
    "doc-1": "Python dictionaries are fast. Python is expressive!",
    "doc-2": "Fast lookup is a dictionary strength.",
}
index = build_inverted_index(documents)

assert index["python"] == {"doc-1": [0, 4]}
assert index["fast"] == {"doc-1": [3], "doc-2": [0]}
assert index["is"] == {"doc-1": [5], "doc-2": [2]}
assert "expressive" in index
assert "expressive!" not in index


**Complexity:** O(total tokens) average time, plus storage proportional to the number of indexed occurrences.


---
## Problem 5 — Invert a Mapping Without Losing Collisions

**Difficulty:** Intermediate  
**Core operations:** `setdefault`, membership

A naïve inversion loses data when multiple keys share a value:

```python
{value: key for key, value in original.items()}
```

Implement `invert_many(mapping)` so every original key is preserved:

```python
{"alice": "admin", "bob": "viewer", "carol": "admin"}
# becomes
{"admin": ["alice", "carol"], "viewer": ["bob"]}
```

Preserve original key order inside each list.


In [7]:
def invert_many(mapping: Mapping[K, V]) -> dict[V, list[K]]:
    """Invert a mapping, collecting all keys that share each value."""
    inverted: dict[V, list[K]] = {}
    for original_key, original_value in mapping.items():
        inverted.setdefault(original_value, []).append(original_key)
    return inverted


roles = {"alice": "admin", "bob": "viewer", "carol": "admin"}
assert invert_many(roles) == {
    "admin": ["alice", "carol"],
    "viewer": ["bob"],
}
assert invert_many({}) == {}


---
## Problem 6 — Inventory Merge with Explicit Conflict Policies

**Difficulty:** Advanced  
**Core operations:** membership, `get`, direct assignment

Merge multiple inventory dictionaries where values are non-negative quantities.

Supported policies:

- `"sum"`: add quantities,
- `"replace"`: later dictionaries overwrite earlier values,
- `"max"`: keep the largest quantity seen.

Requirements:

- Reject an unknown policy with `ValueError`.
- Reject negative quantities with `ValueError`.
- Do not mutate inputs.
- Preserve first-seen SKU order.


In [8]:
def merge_inventories(
    inventories: Iterable[Mapping[str, int]],
    *,
    policy: str = "sum",
) -> dict[str, int]:
    """Merge inventory mappings according to an explicit conflict policy."""
    if policy not in {"sum", "replace", "max"}:
        raise ValueError(f"Unsupported policy: {policy!r}")

    merged: dict[str, int] = {}

    for inventory in inventories:
        for sku, quantity in inventory.items():
            if quantity < 0:
                raise ValueError(f"Negative quantity for {sku!r}: {quantity}")

            if sku not in merged:
                merged[sku] = quantity
            elif policy == "sum":
                merged[sku] += quantity
            elif policy == "replace":
                merged[sku] = quantity
            else:  # policy == "max"
                merged[sku] = max(merged[sku], quantity)

    return merged


warehouses = [
    {"A-10": 4, "B-20": 7},
    {"B-20": 3, "C-30": 9},
    {"A-10": 8},
]
assert merge_inventories(warehouses, policy="sum") == {
    "A-10": 12,
    "B-20": 10,
    "C-30": 9,
}
assert merge_inventories(warehouses, policy="replace") == {
    "A-10": 8,
    "B-20": 3,
    "C-30": 9,
}
assert merge_inventories(warehouses, policy="max") == {
    "A-10": 8,
    "B-20": 7,
    "C-30": 9,
}


---
## Problem 7 — Sparse Vector Dot Product

**Difficulty:** Advanced  
**Core operations:** `get`, membership, `len`

Represent a sparse vector as `{index: value}`. Implement an efficient dot product.

Best-practice requirement: iterate over the **smaller** dictionary and look up matching coordinates in the larger one.

Example:

```python
{0: 2.0, 1000: 3.0} · {0: 4.0, 2: 9.0, 1000: -1.0} == 5.0
```


In [9]:
def sparse_dot(
    left: Mapping[int, float],
    right: Mapping[int, float],
) -> float:
    """Return the dot product of two sparse vectors."""
    if len(left) > len(right):
        left, right = right, left

    total = 0.0
    for index, left_value in left.items():
        total += left_value * right.get(index, 0.0)
    return total


v1 = {0: 2.0, 1000: 3.0}
v2 = {0: 4.0, 2: 9.0, 1000: -1.0}
assert isclose(sparse_dot(v1, v2), 5.0)
assert isclose(sparse_dot({}, v2), 0.0)
assert isclose(sparse_dot({1: 2.5}, {2: 10.0}), 0.0)


**Complexity:** O(min(k₁, k₂)) average time and O(1) extra space.


---
## Problem 8 — Dictionary-Based Multiset Operations

**Difficulty:** Advanced  
**Core operations:** `get`, membership, deletion, dictionary comprehensions

A multiset stores counts instead of mere membership:

```python
{"apple": 2, "pear": 1}
```

Implement:

- `multiset_union`: maximum count per key,
- `multiset_intersection`: minimum positive count for shared keys,
- `multiset_add`: sum counts,
- `multiset_subtract`: subtract and remove keys whose result is zero or negative.

Inputs may omit zero-count keys. Reject negative counts.


In [10]:
def _validate_multiset(multiset: Mapping[K, int]) -> None:
    for key, count in multiset.items():
        if count < 0:
            raise ValueError(f"Negative count for {key!r}: {count}")


def multiset_union(left: Mapping[K, int], right: Mapping[K, int]) -> dict[K, int]:
    _validate_multiset(left)
    _validate_multiset(right)
    result = dict(left)
    for key, count in right.items():
        result[key] = max(result.get(key, 0), count)
    return {key: count for key, count in result.items() if count > 0}


def multiset_intersection(left: Mapping[K, int], right: Mapping[K, int]) -> dict[K, int]:
    _validate_multiset(left)
    _validate_multiset(right)
    if len(left) > len(right):
        left, right = right, left
    result: dict[K, int] = {}
    for key, count in left.items():
        shared = min(count, right.get(key, 0))
        if shared > 0:
            result[key] = shared
    return result


def multiset_add(left: Mapping[K, int], right: Mapping[K, int]) -> dict[K, int]:
    _validate_multiset(left)
    _validate_multiset(right)
    result = dict(left)
    for key, count in right.items():
        result[key] = result.get(key, 0) + count
    return {key: count for key, count in result.items() if count > 0}


def multiset_subtract(left: Mapping[K, int], right: Mapping[K, int]) -> dict[K, int]:
    _validate_multiset(left)
    _validate_multiset(right)
    result = dict(left)
    for key, count in right.items():
        remaining = result.get(key, 0) - count
        if remaining > 0:
            result[key] = remaining
        else:
            result.pop(key, None)
    return result


basket_a = {"apple": 3, "pear": 1, "plum": 2}
basket_b = {"apple": 1, "pear": 4, "banana": 2}

assert multiset_union(basket_a, basket_b) == {
    "apple": 3,
    "pear": 4,
    "plum": 2,
    "banana": 2,
}
assert multiset_intersection(basket_a, basket_b) == {"apple": 1, "pear": 1}
assert multiset_add(basket_a, basket_b) == {
    "apple": 4,
    "pear": 5,
    "plum": 2,
    "banana": 2,
}
assert multiset_subtract(basket_a, basket_b) == {"apple": 2, "plum": 2}


---
## Problem 9 — Transactional Dictionary Patch

**Difficulty:** Advanced  
**Core operations:** `pop`, direct assignment, membership, copying

Apply a sequence of operations atomically. Supported operations:

```python
("set", key, value)
("delete", key)
```

Rules:

- `set` inserts or overwrites.
- `delete` must fail if the key does not exist.
- An unknown operation must fail.
- If any operation fails, the original dictionary must remain unchanged.
- Return a new dictionary; do not mutate the input.


In [11]:
def apply_patch_atomic(
    original: Mapping[K, V],
    operations: Iterable[tuple[Any, ...]],
) -> dict[K, V]:
    """Apply set/delete operations to a copy; commit only if all succeed."""
    working = dict(original)

    for operation in operations:
        if not operation:
            raise ValueError("Operation cannot be empty")

        command = operation[0]
        if command == "set":
            if len(operation) != 3:
                raise ValueError(f"Malformed set operation: {operation!r}")
            _, key, value = operation
            working[key] = value
        elif command == "delete":
            if len(operation) != 2:
                raise ValueError(f"Malformed delete operation: {operation!r}")
            _, key = operation
            removed = working.pop(key, MISSING)
            if removed is MISSING:
                raise KeyError(key)
        else:
            raise ValueError(f"Unknown operation: {command!r}")

    return working


base = {"mode": "safe", "retries": 3}
patched = apply_patch_atomic(
    base,
    [
        ("set", "retries", 5),
        ("set", "timeout", 10),
        ("delete", "mode"),
    ],
)
assert patched == {"retries": 5, "timeout": 10}
assert base == {"mode": "safe", "retries": 3}  # unchanged

try:
    apply_patch_atomic(base, [("set", "x", 1), ("delete", "missing")])
except KeyError:
    pass
else:
    raise AssertionError("Expected KeyError")

assert base == {"mode": "safe", "retries": 3}


**Best-practice note:** copying first is a simple transaction strategy for modest dictionaries. For very large data structures, a change log or persistent data structure may be more appropriate.


---
## Problem 10 — Drain a Dictionary in LIFO Order

**Difficulty:** Intermediate  
**Core operations:** `popitem`, `len`, copying

Implement `drain_lifo(mapping)` that returns key-value pairs from newest insertion to oldest.

Requirements:

- Do not mutate the caller's dictionary.
- Use `popitem()` rather than reversing a list of `.items()`.
- Return a list of `(key, value)` tuples.


In [12]:
def drain_lifo(mapping: Mapping[K, V]) -> list[tuple[K, V]]:
    """Return entries in reverse insertion order without mutating the input."""
    working = dict(mapping)
    drained: list[tuple[K, V]] = []

    while working:
        drained.append(working.popitem())

    return drained


history = {"created": 1, "validated": 2, "published": 3}
assert drain_lifo(history) == [
    ("published", 3),
    ("validated", 2),
    ("created", 1),
]
assert history == {"created": 1, "validated": 2, "published": 3}


`popitem()` is LIFO in current Python versions. It is useful for stack-like workflows, but it does **not** make a dictionary an LRU cache because normal lookups do not move keys to the end.


---
## Problem 11 — Build an Undirected Graph Adjacency Map

**Difficulty:** Advanced  
**Core operations:** `setdefault`, membership, nested mutable values

Given edges `(u, v)`, build:

```python
{node: {neighbor_1, neighbor_2, ...}}
```

Requirements:

- Treat edges as undirected.
- Include nodes that appear only on one side.
- Ignore duplicate edges naturally.
- Reject self-loops.
- Provide a second function that returns each node's degree.


In [13]:
def build_undirected_graph(
    edges: Iterable[tuple[K, K]],
) -> dict[K, set[K]]:
    """Build an adjacency-set representation of a simple undirected graph."""
    graph: dict[K, set[K]] = {}

    for left, right in edges:
        if left == right:
            raise ValueError(f"Self-loop is not allowed: {left!r}")
        graph.setdefault(left, set()).add(right)
        graph.setdefault(right, set()).add(left)

    return graph


def degrees(graph: Mapping[K, set[K]]) -> dict[K, int]:
    return {node: len(neighbors) for node, neighbors in graph.items()}


edges = [
    ("A", "B"),
    ("A", "C"),
    ("B", "C"),
    ("A", "B"),  # duplicate
    ("C", "D"),
]
graph = build_undirected_graph(edges)
assert graph == {
    "A": {"B", "C"},
    "B": {"A", "C"},
    "C": {"A", "B", "D"},
    "D": {"C"},
}
assert degrees(graph) == {"A": 2, "B": 2, "C": 3, "D": 1}


---
## Problem 12 — Layer Configuration and Track Provenance

**Difficulty:** Advanced  
**Core operations:** membership, assignment, `.items()` iteration

Combine configuration layers in increasing priority:

1. defaults,
2. environment,
3. user settings.

Return both:

- the final value for each key,
- the name of the layer that supplied the winning value.

The first appearance of a key determines its position; later overrides must not move it.


In [14]:
def layer_configuration(
    layers: Sequence[tuple[str, Mapping[K, V]]],
) -> tuple[dict[K, V], dict[K, str]]:
    """Overlay configuration layers and record each winning source."""
    values: dict[K, V] = {}
    provenance: dict[K, str] = {}

    for layer_name, layer in layers:
        for key, value in layer.items():
            values[key] = value
            provenance[key] = layer_name

    return values, provenance


layers = [
    ("defaults", {"theme": "light", "timeout": 30, "debug": False}),
    ("environment", {"timeout": 10}),
    ("user", {"theme": "dark", "debug": True}),
]
values, sources = layer_configuration(layers)

assert values == {"theme": "dark", "timeout": 10, "debug": True}
assert list(values) == ["theme", "timeout", "debug"]
assert sources == {"theme": "user", "timeout": "environment", "debug": "user"}


Overwriting an existing key changes its value but does not change its insertion position.


---
## Problem 13 — Deduplicate Records with a Chosen Winner Policy

**Difficulty:** Advanced  
**Core operations:** membership, assignment, dictionary order

Deduplicate records by `id` using one of two policies:

- `"first"`: retain the first record,
- `"last"`: retain the last record's data while preserving the ID's first-seen output position.

Return a list of copied dictionaries so callers cannot mutate the originals through aliases.


In [15]:
def deduplicate_records(
    records: Iterable[Mapping[str, Any]],
    *,
    policy: str = "last",
) -> list[dict[str, Any]]:
    """Deduplicate records by id with deterministic output order."""
    if policy not in {"first", "last"}:
        raise ValueError(f"Unsupported policy: {policy!r}")

    by_id: dict[Hashable, dict[str, Any]] = {}

    for record in records:
        record_id = record["id"]
        if policy == "first" and record_id in by_id:
            continue
        by_id[record_id] = dict(record)

    return list(by_id.values())


records = [
    {"id": 2, "status": "new"},
    {"id": 1, "status": "new"},
    {"id": 2, "status": "processed"},
]
assert deduplicate_records(records, policy="first") == [
    {"id": 2, "status": "new"},
    {"id": 1, "status": "new"},
]
assert deduplicate_records(records, policy="last") == [
    {"id": 2, "status": "processed"},
    {"id": 1, "status": "new"},
]


---
## Problem 14 — Bounded Memoization Cache with Statistics

**Difficulty:** Advanced  
**Core operations:** `get`, membership, `pop`, `len`, insertion order

Create a memoized wrapper with a maximum number of cached entries.

Requirements:

- Cache by one hashable positional argument.
- Count hits and misses.
- Evict the **oldest inserted** key when capacity is exceeded.
- Do not call the wrapped function twice for one cache miss.
- Expose `cache`, `stats()`, and `clear()`.

This is FIFO eviction, not LRU.


In [16]:
@dataclass(frozen=True)
class CacheStats:
    hits: int
    misses: int
    size: int


class FIFOMemoizer:
    def __init__(self, function: Callable[[K], V], maxsize: int = 128) -> None:
        if maxsize <= 0:
            raise ValueError("maxsize must be positive")
        self.function = function
        self.maxsize = maxsize
        self.cache: dict[K, V] = {}
        self._hits = 0
        self._misses = 0

    def __call__(self, argument: K) -> V:
        cached = self.cache.get(argument, MISSING)
        if cached is not MISSING:
            self._hits += 1
            return cached  # type: ignore[return-value]

        self._misses += 1
        value = self.function(argument)
        self.cache[argument] = value

        if len(self.cache) > self.maxsize:
            oldest_key = next(iter(self.cache))
            self.cache.pop(oldest_key)

        return value

    def stats(self) -> CacheStats:
        return CacheStats(self._hits, self._misses, len(self.cache))

    def clear(self) -> None:
        self.cache.clear()
        self._hits = 0
        self._misses = 0


calls: list[int] = []

def expensive_square(number: int) -> int:
    calls.append(number)
    return number * number

memoized_square = FIFOMemoizer(expensive_square, maxsize=2)
assert memoized_square(3) == 9   # miss
assert memoized_square(3) == 9   # hit
assert memoized_square(4) == 16  # miss
assert memoized_square(5) == 25  # miss, evicts 3
assert memoized_square(3) == 9   # miss again

assert calls == [3, 4, 5, 3]
assert memoized_square.stats() == CacheStats(hits=1, misses=4, size=2)
memoized_square.clear()
assert memoized_square.stats() == CacheStats(hits=0, misses=0, size=0)


**Sentinel detail:** a cache may legitimately store `None`, `False`, or `0`, so truthiness checks such as `if cached:` would be incorrect.


---
## Problem 15 — Compute a Dictionary Diff and Reapply It

**Difficulty:** Advanced  
**Core operations:** membership, `get`, direct access, deletion

Implement `dictionary_diff(old, new)` returning:

```python
{
    "added": {key: new_value},
    "removed": {key: old_value},
    "changed": {key: (old_value, new_value)},
    "unchanged": {key: value},
}
```

Then implement `apply_dictionary_diff(old, diff)` to reconstruct `new`.

Preserve `old` key order for removed, changed, and unchanged keys. Added keys should follow their order in `new`.


In [17]:
def dictionary_diff(
    old: Mapping[K, V],
    new: Mapping[K, V],
) -> dict[str, dict[Any, Any]]:
    added: dict[K, V] = {}
    removed: dict[K, V] = {}
    changed: dict[K, tuple[V, V]] = {}
    unchanged: dict[K, V] = {}

    for key, old_value in old.items():
        if key not in new:
            removed[key] = old_value
        elif new[key] != old_value:
            changed[key] = (old_value, new[key])
        else:
            unchanged[key] = old_value

    for key, new_value in new.items():
        if key not in old:
            added[key] = new_value

    return {
        "added": added,
        "removed": removed,
        "changed": changed,
        "unchanged": unchanged,
    }


def apply_dictionary_diff(
    old: Mapping[K, V],
    diff: Mapping[str, Mapping[Any, Any]],
) -> dict[K, V]:
    result = dict(old)

    for key in diff["removed"]:
        result.pop(key, None)

    for key, pair in diff["changed"].items():
        _, new_value = pair
        result[key] = new_value

    for key, value in diff["added"].items():
        result[key] = value

    return result


old = {"a": 1, "b": 2, "c": 3, "e": 5}
new = {"a": 1, "b": 20, "d": 4, "e": 5}
diff = dictionary_diff(old, new)

assert diff == {
    "added": {"d": 4},
    "removed": {"c": 3},
    "changed": {"b": (2, 20)},
    "unchanged": {"a": 1, "e": 5},
}
assert apply_dictionary_diff(old, diff) == new
assert old == {"a": 1, "b": 2, "c": 3, "e": 5}


---
## Problem 16 — Flatten and Unflatten Nested Dictionaries

**Difficulty:** Advanced  
**Core operations:** recursion, membership, direct assignment

Flatten nested dictionaries using tuple paths:

```python
{"db": {"host": "localhost", "port": 5432}}
# becomes
{("db", "host"): "localhost", ("db", "port"): 5432}
```

Then reconstruct the original structure.

Rules:

- Empty nested dictionaries must be preserved.
- Reject path collisions during unflattening.
- Keys must be hashable, but need not be strings.


In [18]:
EMPTY_DICT = object()


def flatten_dict(
    nested: Mapping[K, Any],
    prefix: tuple[K, ...] = (),
) -> dict[tuple[K, ...], Any]:
    """Flatten nested mappings into tuple-key paths."""
    flat: dict[tuple[K, ...], Any] = {}

    if not nested and prefix:
        flat[prefix] = EMPTY_DICT
        return flat

    for key, value in nested.items():
        path = prefix + (key,)
        if isinstance(value, Mapping):
            flat.update(flatten_dict(value, path))
        else:
            flat[path] = value

    return flat


def unflatten_dict(flat: Mapping[tuple[K, ...], Any]) -> dict[K, Any]:
    """Reconstruct nested dictionaries while detecting path collisions."""
    root: dict[K, Any] = {}

    for path, value in flat.items():
        if not path:
            raise ValueError("Paths must contain at least one key")

        current: dict[K, Any] = root
        for part in path[:-1]:
            existing = current.get(part, MISSING)
            if existing is MISSING:
                current[part] = {}
                existing = current[part]
            elif not isinstance(existing, dict):
                raise ValueError(f"Path collision at {part!r}")
            current = existing

        final_key = path[-1]
        if final_key in current:
            raise ValueError(f"Duplicate or colliding path: {path!r}")
        current[final_key] = {} if value is EMPTY_DICT else value

    return root


nested = {
    "db": {"host": "localhost", "port": 5432},
    "features": {},
    "debug": False,
}
flat = flatten_dict(nested)
assert flat[("db", "host")] == "localhost"
assert flat[("features",)] is EMPTY_DICT
assert unflatten_dict(flat) == nested


---
## Problem 17 — Aggregate Event Logs into Nested Metrics

**Difficulty:** Advanced  
**Core operations:** nested `setdefault`, `get`, counting

Each event has:

- `day`
- `user`
- `action`
- optional `duration_ms`

Build:

```python
{
    day: {
        user: {
            "actions": {action: count},
            "total_duration_ms": integer,
            "event_count": integer,
        }
    }
}
```

Requirements:

- Missing duration counts as zero.
- Reject negative durations.
- Use direct indexing for required fields.


In [19]:
def aggregate_events(
    events: Iterable[Mapping[str, Any]],
) -> dict[str, dict[str, dict[str, Any]]]:
    """Aggregate event counts and durations by day and user."""
    result: dict[str, dict[str, dict[str, Any]]] = {}

    for event in events:
        day = event["day"]
        user = event["user"]
        action = event["action"]
        duration = event.get("duration_ms", 0)

        if duration < 0:
            raise ValueError(f"Negative duration: {duration}")

        users = result.setdefault(day, {})
        metrics = users.setdefault(
            user,
            {
                "actions": {},
                "total_duration_ms": 0,
                "event_count": 0,
            },
        )

        actions = metrics["actions"]
        actions[action] = actions.get(action, 0) + 1
        metrics["total_duration_ms"] += duration
        metrics["event_count"] += 1

    return result


events = [
    {"day": "2026-08-01", "user": "u1", "action": "login", "duration_ms": 120},
    {"day": "2026-08-01", "user": "u1", "action": "view", "duration_ms": 40},
    {"day": "2026-08-01", "user": "u1", "action": "view"},
    {"day": "2026-08-01", "user": "u2", "action": "login", "duration_ms": 80},
    {"day": "2026-08-02", "user": "u1", "action": "logout", "duration_ms": 20},
]
aggregated = aggregate_events(events)

assert aggregated["2026-08-01"]["u1"] == {
    "actions": {"login": 1, "view": 2},
    "total_duration_ms": 160,
    "event_count": 3,
}
assert aggregated["2026-08-01"]["u2"]["actions"] == {"login": 1}
assert aggregated["2026-08-02"]["u1"]["total_duration_ms"] == 20


---
## Problem 18 — Diagnose Shared Mutable Defaults

**Difficulty:** Advanced concept  
**Core operations:** `fromkeys`, `setdefault`, identity

Explain and fix this bug:

```python
buckets = dict.fromkeys(["low", "medium", "high"], [])
buckets["low"].append("task-1")
```

Why does every bucket contain `"task-1"`?


In [20]:
# Buggy version: all keys refer to the exact same list object.
buggy = dict.fromkeys(["low", "medium", "high"], [])
buggy["low"].append("task-1")

assert buggy == {
    "low": ["task-1"],
    "medium": ["task-1"],
    "high": ["task-1"],
}
assert buggy["low"] is buggy["medium"] is buggy["high"]

# Correct version 1: create a fresh list for each key.
fixed = {priority: [] for priority in ["low", "medium", "high"]}
fixed["low"].append("task-1")
assert fixed == {"low": ["task-1"], "medium": [], "high": []}
assert fixed["low"] is not fixed["medium"]

# Correct version 2: build groups lazily with setdefault.
lazy: dict[str, list[str]] = {}
for priority, task in [("low", "task-1"), ("high", "task-2")]:
    lazy.setdefault(priority, []).append(task)
assert lazy == {"low": ["task-1"], "high": ["task-2"]}


`dict.fromkeys(keys, value)` reuses the same `value` reference for every key. It is safe for immutable values such as `0`, `None`, or strings, but usually wrong for lists, sets, and dictionaries.


---
## Problem 19 — Safely Remove Expired Sessions

**Difficulty:** Intermediate  
**Core operations:** membership, `pop`, iteration safety

Given `{session_id: expires_at}`, remove entries with `expires_at <= now` and return the removed entries.

Requirements:

- Mutate the supplied dictionary intentionally.
- Do not mutate it while iterating over a live `.items()` view.
- Preserve removal order based on the dictionary's insertion order.


In [21]:
def remove_expired_sessions(
    sessions: MutableMapping[K, int],
    *,
    now: int,
) -> dict[K, int]:
    """Remove and return expired sessions without changing size mid-iteration."""
    expired_keys = [
        session_id
        for session_id, expires_at in sessions.items()
        if expires_at <= now
    ]

    removed: dict[K, int] = {}
    for session_id in expired_keys:
        removed[session_id] = sessions.pop(session_id)

    return removed


sessions = {
    "s1": 100,
    "s2": 250,
    "s3": 150,
    "s4": 400,
}
removed = remove_expired_sessions(sessions, now=200)
assert removed == {"s1": 100, "s3": 150}
assert sessions == {"s2": 250, "s4": 400}


---
## Problem 20 — Reconcile Requested and Available Quantities

**Difficulty:** Advanced  
**Core operations:** `get`, `pop`, membership, deletion semantics

Given an inventory and an order, produce:

- an updated inventory,
- fulfilled quantities,
- backordered quantities.

Rules:

- Quantities must be positive integers.
- A missing SKU has zero availability.
- Remove SKUs from inventory when their remaining quantity becomes zero.
- Do not mutate inputs.


In [22]:
def reconcile_order(
    inventory: Mapping[str, int],
    order: Mapping[str, int],
) -> tuple[dict[str, int], dict[str, int], dict[str, int]]:
    """Fulfill an order as much as possible and report backorders."""
    remaining = dict(inventory)
    fulfilled: dict[str, int] = {}
    backordered: dict[str, int] = {}

    for sku, requested in order.items():
        if not isinstance(requested, int) or isinstance(requested, bool) or requested <= 0:
            raise ValueError(f"Order quantity must be a positive integer for {sku!r}")

        available = remaining.get(sku, 0)
        supplied = min(available, requested)
        shortage = requested - supplied

        if supplied:
            fulfilled[sku] = supplied
            new_quantity = available - supplied
            if new_quantity:
                remaining[sku] = new_quantity
            else:
                remaining.pop(sku, None)

        if shortage:
            backordered[sku] = shortage

    return remaining, fulfilled, backordered


inventory = {"A": 5, "B": 2, "C": 10}
order = {"A": 3, "B": 5, "D": 4}
remaining, fulfilled, backordered = reconcile_order(inventory, order)

assert remaining == {"A": 2, "C": 10}
assert fulfilled == {"A": 3, "B": 2}
assert backordered == {"B": 3, "D": 4}
assert inventory == {"A": 5, "B": 2, "C": 10}


---
# Bonus Worked Examples

These shorter examples reinforce common dictionary patterns.


## Bonus A — Partition Values by a Classifier


In [23]:
def partition_by(
    values: Iterable[T],
    classifier: Callable[[T], K],
) -> dict[K, list[T]]:
    groups: dict[K, list[T]] = {}
    for value in values:
        groups.setdefault(classifier(value), []).append(value)
    return groups


numbers = range(-3, 4)
assert partition_by(numbers, lambda n: "negative" if n < 0 else "zero" if n == 0 else "positive") == {
    "negative": [-3, -2, -1],
    "zero": [0],
    "positive": [1, 2, 3],
}


## Bonus B — Index Records by a Unique Key


In [24]:
def index_unique(
    records: Iterable[Mapping[str, Any]],
    key_name: str,
) -> dict[Hashable, dict[str, Any]]:
    index: dict[Hashable, dict[str, Any]] = {}
    for record in records:
        key = record[key_name]
        if key in index:
            raise ValueError(f"Duplicate {key_name}: {key!r}")
        index[key] = dict(record)
    return index


users = [
    {"id": 10, "name": "Ava"},
    {"id": 11, "name": "Noah"},
]
assert index_unique(users, "id")[11]["name"] == "Noah"


## Bonus C — Rename a Key Safely


In [25]:
def rename_key(
    mapping: Mapping[K, V],
    old_key: K,
    new_key: K,
    *,
    overwrite: bool = False,
) -> dict[K, V]:
    result = dict(mapping)

    value = result.pop(old_key, MISSING)
    if value is MISSING:
        raise KeyError(old_key)
    if not overwrite and new_key in result:
        raise KeyError(f"Destination key already exists: {new_key!r}")

    result[new_key] = value  # renamed key is inserted at the end
    return result


assert rename_key({"a": 1, "b": 2}, "a", "x") == {"b": 2, "x": 1}


## Bonus D — Clear a Shared Dictionary In Place


In [26]:
registry = {"service-a": "healthy", "service-b": "degraded"}
observer_reference = registry

registry.clear()

assert registry == {}
assert observer_reference == {}
assert registry is observer_reference


## Bonus E — Top-K Counts with Stable Tie Breaking


In [27]:
def top_k_counts(counts: Mapping[K, int], k: int) -> list[tuple[K, int]]:
    if k < 0:
        raise ValueError("k cannot be negative")

    # Python sorting is stable. Equal counts retain dictionary insertion order.
    return sorted(counts.items(), key=lambda pair: pair[1], reverse=True)[:k]


counts = {"red": 4, "blue": 7, "green": 7, "yellow": 2}
assert top_k_counts(counts, 3) == [("blue", 7), ("green", 7), ("red", 4)]


---
# Mixed Review Challenge — Complete Solution

Create a compact report from transaction records. Each transaction contains:

- `account`
- `category`
- `amount`
- optional `status`, defaulting to `"posted"`

Return:

```python
{
    "totals_by_account": {account: total_amount},
    "totals_by_category": {category: total_amount},
    "statuses": {status: count},
    "largest_by_account": {account: largest_transaction_record},
}
```

Rules:

- Ignore transactions with status `"cancelled"` from monetary totals.
- Still count every status.
- Preserve first-seen key order.
- For equal largest amounts, retain the earlier transaction.
- Copy retained records.


In [28]:
def transaction_report(
    transactions: Iterable[Mapping[str, Any]],
) -> dict[str, dict[Any, Any]]:
    totals_by_account: dict[Hashable, float] = {}
    totals_by_category: dict[Hashable, float] = {}
    statuses: dict[str, int] = {}
    largest_by_account: dict[Hashable, dict[str, Any]] = {}

    for transaction in transactions:
        account = transaction["account"]
        category = transaction["category"]
        amount = float(transaction["amount"])
        status = transaction.get("status", "posted")

        statuses[status] = statuses.get(status, 0) + 1

        if status == "cancelled":
            continue

        totals_by_account[account] = totals_by_account.get(account, 0.0) + amount
        totals_by_category[category] = totals_by_category.get(category, 0.0) + amount

        previous = largest_by_account.get(account, MISSING)
        if previous is MISSING or amount > float(previous["amount"]):
            largest_by_account[account] = dict(transaction)

    return {
        "totals_by_account": totals_by_account,
        "totals_by_category": totals_by_category,
        "statuses": statuses,
        "largest_by_account": largest_by_account,
    }


transactions = [
    {"account": "A", "category": "books", "amount": 25},
    {"account": "A", "category": "food", "amount": 40, "status": "posted"},
    {"account": "B", "category": "books", "amount": 70, "status": "cancelled"},
    {"account": "B", "category": "food", "amount": 10, "status": "pending"},
    {"account": "A", "category": "food", "amount": 40, "status": "pending"},
]
report = transaction_report(transactions)

assert report["totals_by_account"] == {"A": 105.0, "B": 10.0}
assert report["totals_by_category"] == {"books": 25.0, "food": 90.0}
assert report["statuses"] == {"posted": 2, "cancelled": 1, "pending": 2}
assert report["largest_by_account"]["A"] == {
    "account": "A",
    "category": "food",
    "amount": 40,
    "status": "posted",
}
assert report["largest_by_account"]["B"]["amount"] == 10


# Final Review Questions

1. When should direct indexing be preferred over `get`?
2. Why is a sentinel safer than `None` for some lookups?
3. What does `pop` provide that `del` does not?
4. What order does `popitem()` use in modern Python?
5. Why can `dict.fromkeys(keys, [])` be dangerous?
6. Why should you avoid deleting keys while iterating over `d.items()`?
7. How does `setdefault` simplify nested grouping?
8. Why does assigning to an existing key not move it to the end?
9. When is a shallow dictionary copy insufficient?
10. What is the average-case complexity of lookup, insertion, and deletion?


## Answer Key

1. Use direct indexing when absence is an error and should raise `KeyError`.
2. `None` may be a legitimate stored value; a unique sentinel cannot be confused with user data.
3. `pop` removes and returns the value, and can accept a default for missing keys.
4. LIFO: the most recently inserted item is removed first.
5. Every key references the same mutable list.
6. Changing dictionary size during live iteration raises an error and creates unstable logic.
7. It inserts a default container only for absent keys and returns the existing or new container.
8. Updating a key changes its value, not its original insertion position.
9. When nested mutable objects must be isolated; use `deepcopy` or domain-specific copying.
10. Average-case O(1), with pathological collision cases potentially worse.


# Summary

You have practiced dictionary operations in realistic contexts:

- safe lookup and missing-value semantics,
- frequency counting and grouping,
- nested indexes,
- collision-preserving inversion,
- controlled merges,
- sparse computation,
- multiset arithmetic,
- transactional updates,
- LIFO removal,
- graph representation,
- configuration layering,
- deduplication,
- bounded caching,
- diffs and patches,
- flattening and unflattening,
- event aggregation,
- mutable-default pitfalls,
- safe deletion during iteration,
- inventory reconciliation.

The central design principle is simple: choose the dictionary operation whose failure behavior communicates your intent most clearly.
